In [1]:
import pandas as pd

# Adapts the exact names to the files
df_secondary = pd.read_csv("API_SE.SEC.ENRR_DS2.csv", skiprows=4)
df_unemployment    = pd.read_csv("API_4_DS2.csv",    skiprows=4)
df_expenses   = pd.read_csv("API_SE.XPD.TOTL.GD.ZS_DS2.csv", skiprows=4)

print("✅ Files uploaded")

✅ Files uploaded


In [2]:
# Middle-income countries - a relevant focus for the business question
countries = [
    "Côte d'Ivoire", "Ghana", "Senegal", "Nigeria", "Kenya",
    "Morocco", "Tunisia", "Egypt, Arab Rep.", "Ethiopia", "Tanzania"
]

def clean(df, nom_col):
    df = df[df["Country Name"].isin(countries)]
    df = df.drop(columns=["Country Code", "Indicator Name",
                           "Indicator Code", "Unnamed: 68"], errors="ignore")
    df = df.melt(id_vars=["Country Name"], var_name="Year", value_name=nom_col)
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
    df = df.dropna(subset=["Year", nom_col])
    df["Year"] = df["Year"].astype(int)
    # We're keeping 2000-2023
    df = df[df["Year"].between(2000, 2023)]
    return df

sec  = clean(df_secondary, "Secondary_school_enrollment_rates_%")
unem = clean(df_unemployment, "Youth_unemployment_%")
exp  = clean(df_expenses,   "Education_spending_GDP_%")

print("✅ Cleaning complete")

✅ Cleaning complete


In [3]:
df_final = sec.merge(unem, on=["Country Name", "Year"], how="outer")
df_final = df_final.merge(exp, on=["Country Name", "Year"], how="outer")
df_final = df_final.sort_values(["Country Name", "Year"]).reset_index(drop=True)

print(df_final.head(15))
print(f"\nShape : {df_final.shape}")
print(f"\nMissing values :\n{df_final.isnull().sum()}")

        Country Name  Year  Secondary_school_enrollment_rates_%  \
0   Egypt, Arab Rep.  2000                            77.833946   
1   Egypt, Arab Rep.  2000                            77.833946   
2   Egypt, Arab Rep.  2000                            77.833946   
3   Egypt, Arab Rep.  2000                            77.833946   
4   Egypt, Arab Rep.  2000                            77.833946   
5   Egypt, Arab Rep.  2000                            77.833946   
6   Egypt, Arab Rep.  2000                            77.833946   
7   Egypt, Arab Rep.  2000                            77.833946   
8   Egypt, Arab Rep.  2000                            77.833946   
9   Egypt, Arab Rep.  2000                            77.833946   
10  Egypt, Arab Rep.  2000                            77.833946   
11  Egypt, Arab Rep.  2000                            77.833946   
12  Egypt, Arab Rep.  2000                            77.833946   
13  Egypt, Arab Rep.  2000                            77.83394

In [4]:
df_final.to_csv("worldbank_education_emploi.csv", index=False)
print("✅ Successful export: worldbank_education_emploi.csv")

✅ Successful export: worldbank_education_emploi.csv


In [5]:
# Load the dataset
df = pd.read_csv("worldbank_education_emploi.csv")

# Display variable types
print(df.dtypes)
print("\nShape:", df.shape)

Country Name                            object
Year                                     int64
Secondary_school_enrollment_rates_%    float64
Youth_unemployment_%                   float64
Education_spending_GDP_%               float64
dtype: object

Shape: (18448, 5)


In [6]:
# Count missing values per column
missing = df.isnull().sum()

# Calculate missing percentage
missing_pct = (missing / len(df)) * 100

# Display summary
missing_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
print(missing_df)

                                     Missing Count  Missing %
Country Name                                     0   0.000000
Year                                             0   0.000000
Secondary_school_enrollment_rates_%           4080  22.116219
Youth_unemployment_%                             0   0.000000
Education_spending_GDP_%                      5312  28.794449


In [7]:
# Select numeric columns only
numeric_cols = ["Secondary_school_enrollment_rates_%", "Youth_unemployment_%", "Education_spending_GDP_%"]

# Detect outliers using IQR method
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"{col}: {len(outliers)} outliers | Bounds [{lower:.2f}, {upper:.2f}]")

Secondary_school_enrollment_rates_%: 0 outliers | Bounds [-27.78, 136.36]
Youth_unemployment_%: 2558 outliers | Bounds [-107.45, 208.18]
Education_spending_GDP_%: 666 outliers | Bounds [1.53, 8.18]


In [8]:
# --- CORRECTION 1 : Missing values ---
# Drop rows where all 3 key metric columns are missing (not analyzable)
df = df.dropna(subset=["Secondary_school_enrollment_rates_%", "Youth_unemployment_%", "Education_spending_GDP_%"], how="all")

# Fill remaining missing values with median per country (respects country-specific trends)
for col in numeric_cols:
    df[col] = df.groupby("Country Name")[col].transform(lambda x: x.fillna(x.median()))

# Fill any remaining NaN with overall median (countries with only 1 data point)
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# --- CORRECTION 2 : Outliers (IQR capping) ---
# Cap outliers at IQR bounds to limit distortion without losing data points
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower=lower, upper=upper)

# Verify corrections
print("Missing values after correction:\n", df.isnull().sum())
print("\nShape after correction:", df.shape)

Missing values after correction:
 Country Name                           0
Year                                   0
Secondary_school_enrollment_rates_%    0
Youth_unemployment_%                   0
Education_spending_GDP_%               0
dtype: int64

Shape after correction: (18448, 5)


In [9]:
# --- CONVERT : Year → int32 (lighter, compatible SQL INT and Power BI date axis) ---
df["Year"] = df["Year"].astype("int32")

# --- ARROUND : 3 numeric indicators to 2 decimals for SQL DECIMAL(5,2) and Power BI KPI readability ---
df["Secondary_school_enrollment_rates_%"] = df["Secondary_school_enrollment_rates_%"].round(2)
df["Youth_unemployment_%"] = df["Youth_unemployment_%"].round(2)
df["Education_spending_GDP_%"] = df["Education_spending_GDP_%"].round(2)

# --- CATEGORISE : Year → decade segment for Power BI slicer and SQL GROUP BY ---
df["decade"] = (df["Year"] // 10 * 10).astype(str) + "s"

# Verify final types
print(df.dtypes)
print(df.head(3))

Country Name                            object
Year                                     int32
Secondary_school_enrollment_rates_%    float64
Youth_unemployment_%                   float64
Education_spending_GDP_%               float64
decade                                  object
dtype: object
       Country Name  Year  Secondary_school_enrollment_rates_%  \
0  Egypt, Arab Rep.  2000                                77.83   
1  Egypt, Arab Rep.  2000                                77.83   
2  Egypt, Arab Rep.  2000                                77.83   

   Youth_unemployment_%  Education_spending_GDP_% decade  
0                 59.03                      4.67  2000s  
1                 36.94                      4.67  2000s  
2                  8.98                      4.67  2000s  


In [10]:
# Rename all columns to business-friendly snake_case names
df = df.rename(columns={
    "Country Name": "country",
    "Year": "year",
    "Secondary_school_enrollment_rates_%": "secondary_school_enrollment_rates",
    "Youth_unemployment_%": "youth_unemployment_rates",
    "Education_spending_GDP_%": "education_spending_gdp"
})

# Verify renaming
print(df.columns.tolist())
print(df.head(3))

['country', 'year', 'secondary_school_enrollment_rates', 'youth_unemployment_rates', 'education_spending_gdp', 'decade']
            country  year  secondary_school_enrollment_rates  \
0  Egypt, Arab Rep.  2000                              77.83   
1  Egypt, Arab Rep.  2000                              77.83   
2  Egypt, Arab Rep.  2000                              77.83   

   youth_unemployment_rates  education_spending_gdp decade  
0                     59.03                    4.67  2000s  
1                     36.94                    4.67  2000s  
2                      8.98                    4.67  2000s  


In [11]:
# --- VARIABLE 1 : efficiency_ratio ---
# Ratio : youth unemployment per GDP point spent on education
# Avoid division by zero with replace(0, NaN)
df["efficiency_ratio"] = (
    df["youth_unemployment_rates"] / df["education_spending_gdp"].replace(0, pd.NA)
).round(2)

# --- VARIABLE 2 : schooling_segment ---
# Segment countries by secondary enrollment level (ILO-inspired thresholds)
df["schooling_segment"] = pd.cut(
    df["secondary_school_enrollment_rates"],
    bins=[0, 50, 80, 200],
    labels=["Low", "Medium", "High"]
)

# --- VARIABLE 3 : unemployment_segment ---
# Segment countries by youth unemployment alert level (ILO thresholds)
df["unemployment_segment"] = pd.cut(
    df["youth_unemployment_rates"],
    bins=[0, 15, 30, 100],
    labels=["Low", "Moderate", "Critical"]
)

# --- VARIABLE 4 : education_effort_segment ---
# Segment countries by education budget effort (UNESCO thresholds)
df["education_effort_segment"] = pd.cut(
    df["education_spending_gdp"],
    bins=[0, 3, 5, 100],
    labels=["Low", "Medium", "High"]
)

# --- VARIABLE 5 : unemployment_variation ---
# Year-over-year change in youth unemployment per country
df = df.sort_values(["country", "year"])
df["unemployment_variation"] = df.groupby("country")["youth_unemployment_rates"].diff().round(2)

# Verify all new variables
print(df[["efficiency_ratio", "schooling_segment",
          "unemployment_segment", "education_effort_segment", "unemployment_variation"]].head(10))
print("\nShape:", df.shape)

   efficiency_ratio schooling_segment unemployment_segment  \
0             12.64            Medium             Critical   
1              7.91            Medium             Critical   
2              1.92            Medium                  Low   
3              1.09            Medium                  Low   
4              4.89            Medium             Moderate   
5             44.58            Medium                  NaN   
6              4.70            Medium             Moderate   
7              0.94            Medium                  Low   
8              0.86            Medium                  Low   
9              0.58            Medium                  Low   

  education_effort_segment  unemployment_variation  
0                   Medium                     NaN  
1                   Medium                  -22.09  
2                   Medium                  -27.96  
3                   Medium                   -3.90  
4                   Medium                   17.76  

In [12]:
# Define the final list of columns to keep
final_columns = [
    "country",
    "year",
    "secondary_school_enrollment_rates",
    "youth_unemployment_rates",
    "education_spending_gdp",
    "decade",
    "efficiency_ratio",
    "schooling_segment",
    "unemployment_segment",
    "education_effort_segment",
    "unemployment_variation"
]

# Keep only final columns — no variable is dropped (all are analytically justified)
df_final = df[final_columns].copy()

# Convert category columns to string for SQL and Power BI compatibility
category_cols = ["schooling_segment", "unemployment_segment", "education_effort_segment"]
for col in category_cols:
    df_final[col] = df_final[col].astype(str)

# Replace 'nan' strings (from NaN categories) with None for clean SQL export
df_final[category_cols] = df_final[category_cols].replace("nan", None)

# Final verification
print("Finals columns:", df_final.columns.tolist())
print("Shape final:", df_final.shape)
print("\Preview:\n", df_final.head(5))
print("\nFinals types :\n", df_final.dtypes)

# Export clean dataset for SQL import
df_final.to_csv("worldbank_education_emploi.csv", index=False, encoding="utf-8")
print("\nExport CSV over : worldbank_education_emploi.csv")

Finals columns: ['country', 'year', 'secondary_school_enrollment_rates', 'youth_unemployment_rates', 'education_spending_gdp', 'decade', 'efficiency_ratio', 'schooling_segment', 'unemployment_segment', 'education_effort_segment', 'unemployment_variation']
Shape final: (18448, 11)
\Preview:
             country  year  secondary_school_enrollment_rates  \
0  Egypt, Arab Rep.  2000                              77.83   
1  Egypt, Arab Rep.  2000                              77.83   
2  Egypt, Arab Rep.  2000                              77.83   
3  Egypt, Arab Rep.  2000                              77.83   
4  Egypt, Arab Rep.  2000                              77.83   

   youth_unemployment_rates  education_spending_gdp decade  efficiency_ratio  \
0                     59.03                    4.67  2000s             12.64   
1                     36.94                    4.67  2000s              7.91   
2                      8.98                    4.67  2000s              1.92   
3  

In [13]:
df.to_excel("worldbank_education_emploi.xlsx",index=False,sheet_name="worldbank_education_emploi")

In [14]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

user = "root"
password = "mysql2026"
host = "localhost"
port = 3306
db = "esmel"

engine = create_engine(
    f"mysql+pymysql://{user}:{password}@{host}:{port}/{db}",
    echo=False
)

In [15]:
engine.connect()

In [16]:
df.to_sql(
    name="worldbankeducationemploi",
    con=engine,
    if_exists="replace",
    index=False
)

18448